In [1]:
# preparation
import os

In [2]:
# read json into data frame (very slow)
raw_dir = '/nfs/turbo/twitter-decahose/decahose/raw'
df = sqlContext.read.json(os.path.join(raw_dir,'decahose.2022-03-02.p2.bz2'))

22/11/29 21:03:22 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [20]:
# read tweets
tweet = df.select('created_at','extended_tweet.full_text','lang')
# tweet.printSchema()
tweet.show(10, truncate=True)

+--------------------+--------------------+----+
|          created_at|           full_text|lang|
+--------------------+--------------------+----+
|Wed Mar 02 04:51:...|                null|  ja|
|Wed Mar 02 04:51:...|                null|  es|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|house maids servi...|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
|Wed Mar 02 04:51:...|                null|  en|
+--------------------+--------------------+----+
only showing top 10 rows



In [26]:
# user information
user = df.select('user')
names = df.select('user.id','user.name','user.screen_name', 'user.description')
# names.printSchema()
names.show(10, truncate=True)

+-------------------+--------------------+---------------+-----------------------------------+
|                 id|                name|    screen_name|                        description|
+-------------------+--------------------+---------------+-----------------------------------+
|         1596113701|        (神 ¨̮ 奈)♚✧|  beer_smoke_lv|名古屋のおっぱい代表。 コスプレ ...|
|          581897974|           Majito 🦩|         fgrmr9|               Algún día ⏳\n\n\n...|
|1139392692561932288|      ᴮᴱMy Euphoria⁷|sweetmaknaekook|               The genre is BTS....|
|1341823115836477445|VTuberTweeter | N...|  VTuberTweeter|               ⭐️I auto-retweet ...|
|1480247343676944385|      Jennifer Adams| Ultimatetupman|               Those that have g...|
|1398602686187065344|عايشه عاملات منزل...|1drDqyDU2DHgTf0|               ‏‏‏‏‏عايشه عاملات...|
|1200661454740951040|  camilohernandez012|camiloh54690462|                               null|
|1478253457131847683|     Metro Sports KC| MetroSports_KC|          

In [28]:
# read filtered tweets from sbatch
filter_dir="/nfs/turbo/seas-zhukai/phenology/Twitter/"
# category="antihistamine"
category="pollen"
year="2022"
month="02"
day="01"
df = sqlContext.read.json(os.path.join(filter_dir,"query/",category+"/","Spark/",year+"/",month+"/", day+"/"))

In [7]:
# read tweets with user info
df_sel = df.select('created_at','user.id','user.screen_name','user.description','extended_tweet.full_text','lang') # screen_name appears to be the handle
df_sel = df_sel.filter(df_sel['full_text'].isNotNull())
df_sel.show(10, truncate=True)

+--------------------+-------------------+---------------+--------------------+--------------------+----+
|          created_at|                 id|    screen_name|         description|           full_text|lang|
+--------------------+-------------------+---------------+--------------------+--------------------+----+
|Wed Feb 02 00:01:...|           26069433| BereniceHealey|Memorial Device A...|Is there an early...|  en|
|Tue Feb 01 16:21:...|          365732416| tony_thetiger3|could be fun (he/...|HEPA filters are ...|  en|
|Tue Feb 01 18:30:...|          899545604|       Ayledain|Scientifique, (Dr...|@JLGagnaire @Cerc...|  fr|
|Tue Feb 01 21:17:...|1138875506655420416|      Labor2408|MTA im Labor - Po...|@ladybundlebrent ...|  de|
|Tue Feb 01 22:47:...|1216761854011367427|TheVocalSausage|#Fibrowarrior #Pr...|@duchess_salty @P...|  en|
|Tue Feb 01 20:56:...|           79745383|DowlingWildlife|UK Amateur Natura...|@maybemerlot This...|  en|
|Tue Feb 01 21:00:...|          859133995|Melb

In [8]:
df_sel.count()

67

# Cleaning

In [9]:
df_lang = df_sel.filter(df_sel['lang']=="en")

In [11]:
# opening the file in read mode
# reading the file
# replacing end of line('/n') with ' ' and
# splitting the text it further when '.' is seen.
keywords = open(os.path.join(filter_dir,"keywords/",category+".txt"), "r").read().split("\n")
keywords[0:5]

['pollen']

In [12]:
#df_word_one=df_sel.filter(df_sel['full_text'].contains('Tavist'))
#df_word_one=df_sel.filter(df_sel['full_text'].rlike(r'\bTavistock\b'))
#df_word_one=df_sel.filter(df_sel['full_text'].rlike(r'\bTavist\b'))
df_word_one=df_lang.filter(df_sel['full_text'].rlike(r'\bBenadryl\b'))
df_word_one.show(5)
#df_word_one.count()

+----------+---+-----------+-----------+---------+----+
|created_at| id|screen_name|description|full_text|lang|
+----------+---+-----------+-----------+---------+----+
+----------+---+-----------+-----------+---------+----+



In [25]:
df_word = None
for keyword in keywords:
    df_word_one = df_lang.filter(df_lang['full_text'].rlike('\\b(?i)'+keyword+'\\b')) # word boundary # case insensitive
    if not df_word:
        df_word = df_word_one
    else:
        df_word = df_word.union(df_word_one)
df_word=df_word.distinct()
df_word.show(10,truncate=True)
df_word.count()

+--------------------+-------------------+---------------+--------------------+--------------------+----+
|          created_at|                 id|    screen_name|         description|           full_text|lang|
+--------------------+-------------------+---------------+--------------------+--------------------+----+
|Tue Feb 01 16:21:...|          365732416| tony_thetiger3|could be fun (he/...|HEPA filters are ...|  en|
|Wed Feb 02 00:01:...|           26069433| BereniceHealey|Memorial Device A...|Is there an early...|  en|
|Tue Feb 01 22:36:...|          467465583|   jeni_parsons|lefty priest & pe...|@troutman831 Love...|  en|
|Wed Feb 02 00:30:...| 839611039700381696|SaraGreathouse1|disabled Muslim s...|Along those lines...|  en|
|Tue Feb 01 23:18:...|         2401263991|      pollenkat|Pollen scientist,...|Been scanning pol...|  en|
|Tue Feb 01 17:51:...|1450468585365725196| Higher__Events|Solardo presents ...|HIGHER Dubrovnik ...|  en|
|Tue Feb 01 19:46:...|          450673628|    

40

In [26]:
df_word.select("full_text").show(10,truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|full_text                                                                                                                                                                                                                                                                                                                     |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|HEPA filters are also good for:\n\n-

In [15]:
# write to csv
df_word.write.option("header",True).option("delimiter","\t").mode("overwrite").csv(os.path.join(filter_dir,"output/",category+"/","CSV/",year+"/",month+"/", day+"/"))
# can then read as pd dataframe in regular Jupyter notebook

In [6]:
filter_dir="/nfs/turbo/seas-zhukai/phenology/Twitter/"
category="pollen"
year="2022"
# month="02"
day="05"
keywords = open(os.path.join(filter_dir,"keywords/",category+".txt"), "r").read().split("\n")

month_list=[str(i).zfill(2) for i in range(1,11)]
for month in month_list:
    df = sqlContext.read.json(os.path.join(filter_dir,"query/",category+"/","Spark/",year+"/",month+"/", day+"/"))
    df_sel = df.select('created_at','user.id','user.screen_name','user.description','extended_tweet.full_text','lang') # screen_name appears to be the handle
    df_sel = df_sel.filter(df_sel['full_text'].isNotNull())
    df_lang = df_sel.filter(df_sel['lang']=="en")
    df_word = None
    for keyword in keywords:
        df_word_one = df_lang.filter(df_lang['full_text'].rlike('\\b(?i)'+keyword+'\\b')) # word boundary # case insensitive
        if not df_word:
            df_word = df_word_one
        else:
            df_word = df_word.union(df_word_one)
    df_word=df_word.distinct()
    df_word.write.option("header",True).option("delimiter","\t").mode("overwrite").csv(os.path.join(filter_dir,"query/",category+"/","CSV/",year+"/",month+"/", day+"/"))


AnalysisException: Unable to infer schema for JSON. It must be specified manually.

In [27]:
climwords=["climat", "warm", "early"]
df_CC = None
for climword in climwords:
    df_CC_one = df_word.filter(df_word['full_text'].rlike('(?i)'+climword)) # word boundary # case insensitive
    if not df_CC:
        df_CC = df_CC_one
    else:
        df_CC = df_CC.union(df_CC_one)
df_CC=df_CC.distinct()
df_CC.select("full_text").show(10,truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|full_text                                                                                                                                                                                          |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Is there an early hayfever allergen going around because of the unseasonably warm weather? Had all the sneezing and wheezing today (and a negative test FWIW). Tree pollen starts this time of year|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [30]:
allerwords=["allerg", "hay fever", "hayfever", "rhinitis", "asthma"]
df_aller = None
for allerword in allerwords:
    df_aller_one = df_word.filter(df_word['full_text'].rlike('(?i)'+allerword)) # word boundary # case insensitive
    if not df_aller:
        df_aller = df_aller_one
    else:
        df_aller = df_aller.union(df_aller_one)
df_aller=df_aller.distinct()
df_aller.select("full_text").show(10,truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|full_text                                                                                                                                                                                                                                                                                               |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Please send happy thoughts as I'm being 🚑 to ER... severe allergic reaction from folding grocery bags 